In [1]:
# Berechne die Total Variation Distance für die vier Methoden anhand der Datensätze
import numpy as np
import pandas as pd

from efficient_probit_regression.total_variation_distance import total_variation_distance
from efficient_probit_regression.sampling import compute_leverage_scores, calculate_lewis_weights_exact, calculate_l2_lp_leverage_score, compute_random_evaluations_probabilities_v2, to_density

print("Total Variation Distance")

# load all datasets

%store -r Webspam
%store -r Iris
%store -r KDDCup
%store -r Covertype
%store -r Example2D

p = [1.0, 1.3, 1.5, 1.7, 2.0]

def tvd_table_for_dataset(
    X: np.ndarray,
    dataset_name: str,
    p_values,
    *,
    T_lewis: int = 20,
    m_random: int = 1000,
    rng=2026,
) -> pd.DataFrame:
    """
    Erzeugt einen DataFrame mit allen 6 paarweisen TVDs der 4 Methoden
    (lev, lewis, l2_lp, random) für einen Datensatz und alle p-Werte.
    """

    rows = []

    for p in p_values:
        lev = compute_leverage_scores(X, p=p)
        lw  = calculate_lewis_weights_exact(X, p=p, T=T_lewis)
        l2lp = calculate_l2_lp_leverage_score(X, p=p)[1]
        rnd = compute_random_evaluations_probabilities_v2(X, m=m_random, p=p, rng=rng)

        # Dichten vorbereiten
        lev_d = to_density(lev)
        lw_d  = to_density(lw)

        # 6 Paar-Vergleiche
        tvd_lev_lewis   = round(total_variation_distance(lev, lw, normalize=True), 4)
        tvd_lev_l2_lp   = round(total_variation_distance(lev_d, l2lp), 4)
        tvd_lev_random  = round(total_variation_distance(lev_d, rnd), 4)
        tvd_l2_lp_lewis = round(total_variation_distance(l2lp, lw_d), 4)
        tvd_lewis_rand  = round(total_variation_distance(lw_d, rnd), 4)
        tvd_l2_lp_rand  = round(total_variation_distance(l2lp, rnd), 4)

        rows.append({
            "dataset": dataset_name,
            "p": float(p),
            "lev - lewis": float(tvd_lev_lewis),
            "lev - l2lp": float(tvd_lev_l2_lp),
            "lev - random": float(tvd_lev_random),
            "l2lp - lewis": float(tvd_l2_lp_lewis),
            "lewis - random": float(tvd_lewis_rand),
            "l2lp - random": float(tvd_l2_lp_rand),
        })

    df = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
    return df

datasets = {
    "Iris": Iris.X,
    "Example2D": Example2D.X,
    "Webspam": Webspam.X,
    "KDDCup": KDDCup.X,
    "Covertype": Covertype.X,
}

tvd_dfs = {}

for name, X in datasets.items():
    df = tvd_table_for_dataset(
        X,
        dataset_name=name,
        p_values=p,
        T_lewis=20,
        m_random=1000,
        rng=2026
    )
    tvd_dfs[name] = df

    print(f"\n=== {name} ===")
    display(df)

Total Variation Distance

=== Iris ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Iris,1.0,0.1241,0.1463,0.1353,0.0766,0.0987,0.1479
1,Iris,1.3,0.1383,0.1360,0.2259,0.0626,0.1205,0.1575
2,Iris,1.5,0.2215,0.2241,0.2874,0.0448,0.1334,0.1543
3,Iris,1.7,0.2059,0.2088,0.2106,0.0295,0.1452,0.1610
4,Iris,2.0,0.0000,0.0000,0.1614,0.0000,0.1614,0.1614



=== Example2D ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Example2D,1.0,0.2196,0.2366,0.2877,0.1120,0.0899,0.1077
1,Example2D,1.3,0.1442,0.1286,0.1526,0.0682,0.1173,0.1295
2,Example2D,1.5,0.2043,0.2073,0.2797,0.0610,0.1361,0.1260
3,Example2D,1.7,0.2724,0.2516,0.3080,0.0351,0.1548,0.1530
4,Example2D,2.0,0.0000,0.0000,0.1825,0.0000,0.1825,0.1825



=== Webspam ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Webspam,1.0,0.1048,0.1398,0.2766,0.0992,0.2664,0.3094
1,Webspam,1.3,0.1113,0.1688,0.2730,0.1109,0.2713,0.3373
2,Webspam,1.5,0.1981,0.2176,0.3624,0.0841,0.2734,0.3201
3,Webspam,1.7,0.2211,0.2245,0.4113,0.0523,0.2745,0.3019
4,Webspam,2.0,0.0000,0.0000,0.3017,0.0000,0.3017,0.3017



=== KDDCup ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,KDDCup,1.0,0.1606,0.2232,0.4868,0.1307,0.5475,0.5945
1,KDDCup,1.3,0.1803,0.2060,0.5314,0.0901,0.5003,0.5271
2,KDDCup,1.5,0.2624,0.2822,0.4900,0.0613,0.4479,0.4636
3,KDDCup,1.7,0.3596,0.3510,0.5509,0.0403,0.3736,0.3879
4,KDDCup,2.0,0.0000,0.0000,0.2945,0.0000,0.2945,0.2945



=== Covertype ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Covertype,1.0,0.1949,0.1730,0.2119,0.0633,0.3727,0.3472
1,Covertype,1.3,0.1938,0.1964,0.2619,0.0533,0.3856,0.3938
2,Covertype,1.5,0.2024,0.2108,0.3888,0.0420,0.3918,0.4010
3,Covertype,1.7,0.2871,0.2931,0.3842,0.0355,0.3968,0.4072
4,Covertype,2.0,0.0000,0.0000,0.4087,0.0000,0.4087,0.4087


In [2]:
import pandas as pd
from pathlib import Path

df_all = pd.concat(tvd_dfs.values(), ignore_index=True)

# fürs Anzeigen optional runden: alle Spalten außer dataset und p
df_all_show = df_all.copy()
value_cols = [c for c in df_all_show.columns if c not in ("dataset", "p")]

for c in value_cols:
    df_all_show[c] = df_all_show[c].astype(float).round(8)

print("\n=== ALL DATASETS (combined) ===")
display(df_all_show)

out_dir = Path("Tabellen")
out_dir.mkdir(exist_ok=True, parents=True)
out_csv = "tvd_all_methods.csv"
df_all.to_csv(out_dir / out_csv, index=False, float_format="%.10f")
print("Saved:", out_csv)


=== ALL DATASETS (combined) ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Iris,1.0,0.1241,0.1463,0.1353,0.0766,0.0987,0.1479
1,Iris,1.3,0.1383,0.1360,0.2259,0.0626,0.1205,0.1575
2,Iris,1.5,0.2215,0.2241,0.2874,0.0448,0.1334,0.1543
3,Iris,1.7,0.2059,0.2088,0.2106,0.0295,0.1452,0.1610
4,Iris,2.0,0.0000,0.0000,0.1614,0.0000,0.1614,0.1614
5,Example2D,1.0,0.2196,0.2366,0.2877,0.1120,0.0899,0.1077
6,Example2D,1.3,0.1442,0.1286,0.1526,0.0682,0.1173,0.1295
7,Example2D,1.5,0.2043,0.2073,0.2797,0.0610,0.1361,0.1260
8,Example2D,1.7,0.2724,0.2516,0.3080,0.0351,0.1548,0.1530
9,Example2D,2.0,0.0000,0.0000,0.1825,0.0000,0.1825,0.1825


Saved: tvd_all_methods.csv
